<a href="https://colab.research.google.com/github/zilongcheng582-cyber/atayal-nlp-summer/blob/main/notebooks/03_lstm_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 LSTM Language Model — Atayal (Zeyuan Dialect)
Character-level LSTM language model built with PyTorch.

## 模型结构
- Embedding dim: 32
- Hidden size: 128
- Num layers: 2        ← LSTM的层数
- Linear层: 1          ← 输出层
- Total Parameters: 224,861

## 训练结果
- Epochs: 10
- Final Loss: 0.3163
- 生成示例: "ga musa mlyap utux tlyapan myan ga..."


In [3]:
import torch
import torch.nn as nn
import urllib.request

# 下载数据
for i in range(1, 13):
    url = f"https://raw.githubusercontent.com/zilongcheng582-cyber/atayal-nlp-summer/main/data/raw/{i}.pyu"
    urllib.request.urlretrieve(url, f"{i}.pyu")

print("数据下载完成！")
print(f"PyTorch 版本：{torch.__version__}")

数据下载完成！
PyTorch 版本：2.10.0+cpu


In [4]:
# 数据准备
def has_chinese(text):
    return any('\u4e00' <= char <= '\u9fff' for char in text)

all_lines = []
for i in range(1, 13):
    with open(f"{i}.pyu", "r", encoding="utf-8") as f:
        lines = f.readlines()
    lines = [line.strip() for line in lines if line.strip()]
    all_lines.extend(lines)

clean_lines = [line for line in all_lines if not has_chinese(line)]
full_text = "\n".join(clean_lines)

# 字符集
chars = sorted(set(full_text))
char2idx = {char: idx for idx, char in enumerate(chars)}
idx2char = {idx: char for idx, char in enumerate(chars)}

VOCAB_SIZE = len(chars)
print(f"字符集大小：{VOCAB_SIZE}")

# 编码
encoded = [char2idx[c] for c in full_text]
print(f"总字符数：{len(encoded)}")

字符集大小：61
总字符数：58437


In [5]:
class AtayalLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.fc(x)
        return x

# 初始化模型
model = AtayalLSTM(
    vocab_size=VOCAB_SIZE,  # 61
    embed_dim=32,
    hidden_size=128,
    num_layers=2
)

print(model)
print(f"\n模型参数总数：{sum(p.numel() for p in model.parameters())}")

AtayalLSTM(
  (embedding): Embedding(61, 32)
  (lstm): LSTM(32, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=61, bias=True)
)

模型参数总数：224861


In [6]:
import torch

SEQ_LEN = 50
BATCH_SIZE = 32

# 切成训练对
inputs = []
targets = []

for i in range(0, len(encoded) - SEQ_LEN):
    inputs.append(encoded[i : i + SEQ_LEN])
    targets.append(encoded[i + 1 : i + SEQ_LEN + 1])

# 转成 PyTorch tensor
inputs = torch.tensor(inputs, dtype=torch.long)
targets = torch.tensor(targets, dtype=torch.long)

print(f"inputs shape：{inputs.shape}")
print(f"targets shape：{targets.shape}")

inputs shape：torch.Size([58387, 50])
targets shape：torch.Size([58387, 50])


In [7]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"总样本数：{len(dataset)}")
print(f"每个batch大小：{BATCH_SIZE}")
print(f"总batch数：{len(dataloader)}")

总样本数：58387
每个batch大小：32
总batch数：1825


In [8]:
import torch.optim as optim

# 训练设置
EPOCHS = 10
LEARNING_RATE = 0.003

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 训练循环
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_inputs, batch_targets in dataloader:
        optimizer.zero_grad()
        output = model(batch_inputs)
        loss = criterion(
            output.view(-1, VOCAB_SIZE),
            batch_targets.view(-1)
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}")

Epoch 1/10 - Loss: 1.4264
Epoch 2/10 - Loss: 0.8068
Epoch 3/10 - Loss: 0.5547
Epoch 4/10 - Loss: 0.4399
Epoch 5/10 - Loss: 0.3868
Epoch 6/10 - Loss: 0.3598
Epoch 7/10 - Loss: 0.3429
Epoch 8/10 - Loss: 0.3318
Epoch 9/10 - Loss: 0.3231
Epoch 10/10 - Loss: 0.3163


In [9]:
def generate_text(model, start_str, length=100):
    model.eval()
    chars = [char2idx[c] for c in start_str]

    generated = start_str
    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([chars[-SEQ_LEN:]], dtype=torch.long)
            output = model(x)
            pred = output[0, -1, :].argmax().item()
            generated += idx2char[pred]
            chars.append(pred)

    return generated

# 用泰雅语开头生成文字
result = generate_text(model, "ga ", length=100)
print(result)


ga musa mlyap utux tlyapan myan ga, mutuw psbbay na pin’sugan ’sinuw ku lping hiya ga, bhci ru glabang 
